# Initialization

In [0]:
import pyspark.sql.functions as f
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

### Read Bronze table

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")

In [0]:
df.display()

# Silver Transformations

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Customer ID Cleanup

In [0]:
df = df.withColumn(
    "CID",
    f.when(col("CID").startswith("NAS"), f.substring(col("CID"), 4, f.length(col("CID"))))
    .otherwise(col("CID"))
)

### Birthdate Validation

In [0]:
df = df.withColumn(
    "bdate",
    f.when(col("bdate") > f.current_date(), None)
     .otherwise(col("bdate"))
)

### Gender Normalization

In [0]:
df.groupBy("GEN").count().show()

In [0]:
df = df.withColumn(
    "gen",
    f.when(f.upper(col("gen")).isin("F", "FEMALE"), "Female")
     .when(f.upper(col("gen")).isin("M", "MALE"), "Male")
     .otherwise("n/a")
)

### Renaming Columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"
}
for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### Sanity checks of dataframe

In [0]:
df.limit(10).display()

# Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

### Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customers LIMIT 10